# 1Day Start 役職推定パイプライン

複数の試合データを混ぜずに、各データセットごとに個別学習してから最後にアンサンブルします。

- dataset ごとに独立学習
- fold 分割は game 単位で実施
- 検証時は OOF + 他 dataset の固定モデルでリークを避ける
- 最終予測は dataset 間アンサンブル

In [ ]:
import sys
import json
import time
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import xgboost as xgb
import optuna
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from src.pipelines.training_pipeline import load_training_config, get_data_paths
from src.Rolepredicter.role_assignment import assign_roles_for_non_seer, assign_roles_for_seer_by_divination

print(pd.DataFrame([
    {"key": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
    {"key": "CONFIG_PATH", "value": str(CONFIG_PATH)},
    {"key": "python", "value": sys.version.split()[0]},
]))

            key                                              value
0  PROJECT_ROOT  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
1   CONFIG_PATH  c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
2        python                                             3.13.1


C:\Users\takic\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_OPTIONS = {
    "day_filter": 0,
    "day2_flag": False,
    "n_trials": 50,
    "top_k_features": 10,
}

config = load_training_config(CONFIG_PATH)
if "n_trials" in config:
    RUN_OPTIONS["n_trials"] = int(config["n_trials"])

summary_df = pd.DataFrame([
    {"item": "day_filter", "value": RUN_OPTIONS["day_filter"]},
    {"item": "day2_flag", "value": RUN_OPTIONS["day2_flag"]},
    {"item": "n_trials", "value": RUN_OPTIONS["n_trials"]},
    {"item": "data_paths", "value": len(config.get("data_paths", []))},
])
summary_df

,item,value
0,day_filter,0
1,day2_flag,False
2,n_trials,20
3,data_paths,2


In [3]:
data_paths = get_data_paths(config)

requested_day = int(RUN_OPTIONS["day_filter"])
dataset_frames = []
feature_columns = []
all_role_values = []
used_days = []

base_drop_cols = [
    "id", "role", "source_file", "dataset_tag", "day", "role_encoded",
    "Est_id_Fact_role", "Est_id_Est_roles", "character_name",
    "agent_name", "combined_text", "seer_co_order"
]
meta_raw_cols = [
    "True_Div_result_1", "True_Div_recepient_id_1",
    "True_Div_result_2", "True_Div_recepient_id_2",
    "exec_id", "attack_id"
]
day2_col = []
id_like_cols = []
flag_like_cols = []
drop_cols = list(set(base_drop_cols + meta_raw_cols + day2_col + id_like_cols + flag_like_cols))

for p in data_paths:
    p_path = Path(p)
    df_raw = pd.read_csv(p)

    if "day" not in df_raw.columns:
        print(f"skip {p_path.name}: day column not found")
        continue

    df_day = df_raw[df_raw["day"] == requested_day].copy()
    used_day = requested_day

    if df_day.empty:
        available_days = sorted(pd.Series(df_raw["day"]).dropna().unique().tolist())
        if available_days:
            used_day = int(available_days[0])
            df_day = df_raw[df_raw["day"] == used_day].copy()
            print(f"loaded {p_path.name}: raw={len(df_raw)} day={len(df_day)} (fallback day={used_day}, requested={requested_day})")
        else:
            print(f"loaded {p_path.name}: raw={len(df_raw)} day=0 (no valid day values)")
    else:
        print(f"loaded {p_path.name}: raw={len(df_raw)} day={len(df_day)} (day={used_day})")

    if df_day.empty:
        continue

    df_day["dataset_tag"] = p_path.stem
    dataset_frames.append({
        "dataset_tag": p_path.stem,
        "df": df_day,
        "used_day": used_day,
        "source_path": str(p_path),
    })
    used_days.append(used_day)
    all_role_values.extend(df_day["role"].astype(str).tolist())

    X_df_tmp = df_day.drop(columns=drop_cols, errors="ignore")
    for col in X_df_tmp.columns:
        if col not in feature_columns:
            feature_columns.append(col)

if not dataset_frames:
    raise ValueError("day フィルタ後のデータが 0 件です。training_config.json の data_paths または day 列を確認してください。")

label_encoder = LabelEncoder()
label_encoder.fit(sorted(pd.Series(all_role_values).dropna().astype(str).unique().tolist()))

final_features = feature_columns
bundle_rows = []
dataset_bundles = []

for item in dataset_frames:
    df = item["df"].copy()
    dataset_tag = item["dataset_tag"]

    X_df = df.drop(columns=drop_cols, errors="ignore").reindex(columns=final_features, fill_value=0)
    X = X_df.apply(pd.to_numeric, errors="coerce").fillna(0).values
    y = label_encoder.transform(df["role"].astype(str).values)
    agent_names = df["agent_name"].astype(str).values if "agent_name" in df.columns else np.array(["unknown"] * len(df))

    meta = {
        "source_file": df["source_file"].values,
        "dataset_tag": df["dataset_tag"].values if "dataset_tag" in df.columns else np.array([dataset_tag] * len(df)),
        "div_result1": df["True_Div_result_1"].values if "True_Div_result_1" in df.columns else np.full(len(df), np.nan),
        "div_id1": df["True_Div_recepient_id_1"].values if "True_Div_recepient_id_1" in df.columns else np.full(len(df), np.nan),
        "div_result2": df["True_Div_result_2"].values if "True_Div_result_2" in df.columns else np.full(len(df), np.nan),
        "div_id2": df["True_Div_recepient_id_2"].values if "True_Div_recepient_id_2" in df.columns else np.full(len(df), np.nan),
        "exec_id": df["exec_id"].values if "exec_id" in df.columns else np.full(len(df), np.nan),
        "attack_id": df["attack_id"].values if "attack_id" in df.columns else np.full(len(df), np.nan),
    }

    dataset_bundles.append({
        "dataset_tag": dataset_tag,
        "df": df,
        "X": X,
        "X_df": X_df,
        "y": y,
        "meta": meta,
        "agent_names": agent_names,
        "source_files": meta["source_file"],
        "final_features": final_features,
        "used_day": item["used_day"],
        "source_path": item["source_path"],
    })

    unique_agents = sorted(pd.Series(agent_names).dropna().astype(str).unique().tolist())
    bundle_rows.append({
        "dataset_tag": dataset_tag,
        "samples": len(X),
        "features": len(final_features),
        "games": len(np.unique(meta["source_file"])),
        "unique_agents": len(unique_agents),
        "roles": ", ".join(label_encoder.classes_),
        "used_day": item["used_day"],
        "source_path": item["source_path"],
    })

bundle_summary_df = pd.DataFrame(bundle_rows)

if used_days:
    RUN_OPTIONS["day_filter"] = int(min(used_days))

print(f"Loaded dataset bundles: {len(dataset_bundles)}")
display(bundle_summary_df)
print(f"Final shared feature count: {len(final_features)}")
print(f"Roles: {list(label_encoder.classes_)}")

✓ Data files selected:
  - all_feature_table_2025sp17_with_talks.csv
  - all_feature_table_2025summer_with_talks2.csv
loaded all_feature_table_2025sp17_with_talks.csv: raw=1060 day=530 (fallback day=1, requested=0)
loaded all_feature_table_2025summer_with_talks2.csv: raw=1190 day=595 (fallback day=1, requested=0)
Loaded dataset bundles: 2


,dataset_tag,samples,features,games,unique_agents,roles,used_day,source_path
0,all_feature_table_2025sp17_with_talks,530,217,106,9,"POSSESSED, SEER, VILLAGER, WEREWOLF",1,c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...
1,all_feature_table_2025summer_with_talks2,595,217,119,8,"POSSESSED, SEER, VILLAGER, WEREWOLF",1,c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定...


Final shared feature count: 217
Roles: [np.str_('POSSESSED'), np.str_('SEER'), np.str_('VILLAGER'), np.str_('WEREWOLF')]


In [ ]:
import xgboost as xgb
import optuna
from collections import Counter
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("=" * 70)
print("1DAY START モデル訓練 (dataset別学習 + OOF ensemble)")
print("=" * 70)

t0 = time.time()
predictor = None
run_status = {"ok": False, "error": None, "elapsed_sec": None}

views = ["SEER", "WEREWOLF", "VILLAGER", "POSSESSED"]
ensemble_weights_by_view = {}
bundle_results = []
cv_results = []
models_by_bundle = {}
oof_proba_by_bundle = {}
ensemble_eval_rows = []

n_splits = max(2, int(RUN_OPTIONS.get("n_splits", 5)))
fold_seed = int(RUN_OPTIONS.get("fold_seed", 42))
fold_search_iter = max(1, int(RUN_OPTIONS.get("fold_search_iter", 30)))
n_trials = max(1, int(RUN_OPTIONS.get("n_trials", 50)))

if "assign_roles_for_non_seer" not in globals() or "assign_roles_for_seer_by_divination" not in globals():
    from src.Rolepredicter.role_assignment import assign_roles_for_non_seer, assign_roles_for_seer_by_divination

try:
    print(f"\n✓ dataset bundles: {len(dataset_bundles)}")
    for bundle in dataset_bundles:
        dataset_tag = bundle["dataset_tag"]
        X_bundle = bundle["X"]
        y_bundle = bundle["y"]
        meta_bundle = bundle["meta"]
        agent_names_bundle = bundle["agent_names"]
        source_files_bundle = bundle["source_files"]
        classes_bundle = np.unique(y_bundle)
        weights = compute_class_weight(class_weight="balanced", classes=classes_bundle, y=y_bundle)
        weight_dict = dict(zip(classes_bundle, weights))
        label_count = len(label_encoder.classes_)

        role_counts = {"POSSESSED": 1, "SEER": 1, "VILLAGER": 2, "WEREWOLF": 1}
        group_keys = np.array(source_files_bundle)
        unique_games = np.unique(group_keys)

        game_agent_counter = {}
        game_pair_counter = {}
        all_agents = set()
        all_pairs = set()

        for g in unique_games:
            idx = np.where(group_keys == g)[0]
            agent_cnt = Counter(agent_names_bundle[i] for i in idx)
            pair_cnt = Counter((agent_names_bundle[i], int(y_bundle[i])) for i in idx)
            game_agent_counter[g] = agent_cnt
            game_pair_counter[g] = pair_cnt
            all_agents.update(agent_cnt.keys())
            all_pairs.update(pair_cnt.keys())

        all_agents = sorted(all_agents)
        all_pairs = sorted(all_pairs)

        def ratio_mse_after_add(fold_counters, add_counter, keys, target_fold):
            total_mse = 0.0
            ideal = 1.0 / n_splits
            eps = 1e-12
            for key in keys:
                vals = []
                for f in range(n_splits):
                    base = fold_counters[f].get(key, 0)
                    vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
                vals = np.array(vals, dtype=float)
                s = vals.sum()
                if s <= eps:
                    continue
                ratios = vals / s
                total_mse += float(np.mean((ratios - ideal) ** 2))
            return total_mse / max(len(keys), 1)

        def variance_after_add(fold_counts, add_counter, keys, target_fold):
            total_var = 0.0
            for key in keys:
                vals = []
                for f in range(n_splits):
                    base = fold_counts[f].get(key, 0)
                    vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
                total_var += float(np.var(vals))
            return total_var / max(len(keys), 1)

        def greedy_assign_once(seed: int):
            rng = np.random.default_rng(seed)
            game_order = list(unique_games)
            rng.shuffle(game_order)

            fold_agent_counts = [Counter() for _ in range(n_splits)]
            fold_pair_counts = [Counter() for _ in range(n_splits)]
            fold_game_counts = [0 for _ in range(n_splits)]
            game_to_fold = {}

            w_agent = 1.35
            w_pair = 0.45
            w_size = 0.2

            def score_fold_with_candidate(fold_idx, game_id):
                agent_score = ratio_mse_after_add(fold_agent_counts, game_agent_counter[game_id], all_agents, fold_idx)
                pair_score = variance_after_add(fold_pair_counts, game_pair_counter[game_id], all_pairs, fold_idx)
                size_vec = np.array(fold_game_counts, dtype=float)
                size_vec[fold_idx] += 1.0
                size_imbalance = float(np.var(size_vec))
                return w_agent * agent_score + w_pair * pair_score + w_size * size_imbalance

            for g in game_order:
                best_fold = 0
                best_score = np.inf
                for f in range(n_splits):
                    s = score_fold_with_candidate(f, g)
                    if s < best_score:
                        best_score = s
                        best_fold = f
                game_to_fold[g] = best_fold
                fold_game_counts[best_fold] += 1
                fold_agent_counts[best_fold].update(game_agent_counter[g])
                fold_pair_counts[best_fold].update(game_pair_counter[g])

            final_score = (
                w_agent * ratio_mse_after_add(fold_agent_counts, Counter(), all_agents, 0)
                + w_pair * variance_after_add(fold_pair_counts, Counter(), all_pairs, 0)
                + w_size * float(np.var(np.array(fold_game_counts, dtype=float)))
            )
            return game_to_fold, fold_agent_counts, fold_pair_counts, fold_game_counts, final_score

        best = None
        best_score = np.inf
        scores_history = []
        for k in range(fold_search_iter):
            out = greedy_assign_once(seed=fold_seed + k)
            scores_history.append(out[4])
            if out[4] < best_score:
                best = out
                best_score = out[4]

        mean_search_score = float(np.mean(scores_history))
        min_search_score = float(np.min(scores_history))
        print(f"\n[{dataset_tag}] Fold search completed:")
        print(f"  Best score: {best_score:.6f}")
        print(f"  Mean score: {mean_search_score:.6f}")
        print(f"  Min score:  {min_search_score:.6f}")

        game_to_fold, fold_agent_counts, fold_pair_counts, fold_game_counts, _ = best
        fold_id_per_row = np.array([game_to_fold[g] for g in group_keys], dtype=int)

        split_indices = []
        for fold in range(n_splits):
            val_idx = np.where(fold_id_per_row == fold)[0]
            train_idx = np.where(fold_id_per_row != fold)[0]
            split_indices.append((train_idx, val_idx))

        fold_summary_df = pd.DataFrame([
            {
                "dataset_tag": dataset_tag,
                "fold": fold + 1,
                "n_samples": int(np.sum(fold_id_per_row == fold)),
                "n_games": int(len(np.unique(group_keys[fold_id_per_row == fold]))),
            }
            for fold in range(n_splits)
        ])

        fold_role_df = pd.DataFrame([
            {
                "dataset_tag": dataset_tag,
                "fold": fold + 1,
                "role": label_encoder.inverse_transform([r])[0],
                "count": int(np.sum(y_bundle[fold_id_per_row == fold] == r)),
            }
            for fold in range(n_splits)
            for r in np.unique(y_bundle)
        ])

        fold_agent_df = pd.DataFrame([
            {
                "dataset_tag": dataset_tag,
                "fold": fold + 1,
                "agent": str(ag),
                "count": int(np.sum(agent_names_bundle[fold_id_per_row == fold] == ag)),
            }
            for fold in range(n_splits)
            for ag in sorted(np.unique(agent_names_bundle))
        ])

        print("Fold x Role distribution:")
        display(fold_role_df.pivot(index="fold", columns="role", values="count").fillna(0).astype(int))
        print(f"Fold x Agent distribution ({dataset_tag}):")
        display(fold_agent_df.pivot(index="fold", columns="agent", values="count").fillna(0).astype(int))

        def evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val, source_files_val):
            all_p, all_t = [], []
            eval_games = np.unique(source_files_val)

            for game_id in eval_games:
                game_mask = source_files_val == game_id
                game_indices = np.where(game_mask)[0]

                if len(game_indices) < 2:
                    continue

                preds_game = preds_proba[game_indices]
                y_game = y_val[game_indices]
                meta_game = {k: v[game_indices] for k, v in meta_val.items()}

                if view_name == "SEER":
                    res = assign_roles_for_seer_by_divination(
                        preds_game, y_game, role_counts, label_encoder.classes_, label_encoder,
                        meta_game["div_result1"], meta_game["div_id1"],
                        meta_game["div_result2"], meta_game["div_id2"],
                        meta_game["exec_id"], meta_game["attack_id"],
                        RUN_OPTIONS["day2_flag"],
                    )
                    for k in ["black", "white"]:
                        if res[k][0].size > 0:
                            all_p.extend(res[k][0])
                            all_t.extend(res[k][1])
                else:
                    p, t = assign_roles_for_non_seer(
                        preds_game, y_game, role_counts, label_encoder.classes_, label_encoder,
                        view_name, meta_game["exec_id"], meta_game["attack_id"],
                        RUN_OPTIONS["day2_flag"],
                    )
                    if p.size > 0:
                        all_p.extend(p)
                        all_t.extend(t)

            if len(all_t) == 0:
                return 0.0, 0

            target_label = "POSSESSED" if view_name == "WEREWOLF" else "WEREWOLF"
            target_id = label_encoder.transform([target_label])[0]
            score = f1_score(all_t, all_p, labels=[target_id], average="macro", zero_division=0)
            return float(score), len(all_t)

        bundle_models = {}
        bundle_oof = {}
        bundle_view_results = []

        print(f"\n[{dataset_tag}] training start")
        for view in views:
            print(f"\n  {'=' * 65}")
            print(f"  {dataset_tag} / {view} 視点モデル: Optuna探索開始")
            print(f"  {'=' * 65}")

            def objective(trial):
                params = {
                    "objective": "multi:softprob",
                    "num_class": label_count,
                    "eval_metric": "mlogloss",
                    "tree_method": "hist",
                    "random_state": 42,
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                    "n_estimators": trial.suggest_int("n_estimators", 150, 800),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                    "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
                    "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                    "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                }

                fold_scores = []
                for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
                    X_tr, X_val = X_bundle[train_idx], X_bundle[val_idx]
                    y_tr, y_val = y_bundle[train_idx], y_bundle[val_idx]
                    meta_val = {k: v[val_idx] for k, v in meta_bundle.items()}
                    source_files_val = source_files_bundle[val_idx]
                    w_tr = np.array([weight_dict[label] for label in y_tr])

                    model = xgb.XGBClassifier(**params)
                    model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)
                    preds_proba = model.predict_proba(X_val)
                    score, _ = evaluate_view_with_constraints(view, preds_proba, y_val, meta_val, source_files_val)
                    fold_scores.append(score)

                    trial.report(np.mean(fold_scores), step=fold)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

                return float(np.mean(fold_scores))

            study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

            best_params = study.best_params.copy()
            best_params.update({
                "objective": "multi:softprob",
                "num_class": label_count,
                "eval_metric": "mlogloss",
                "tree_method": "hist",
                "random_state": 42,
            })

            print(f"\n    Best trial: {study.best_trial.number}")
            print(f"    Best CV score: {study.best_value:.4f}")
            print(f"    Best params: {study.best_params}")

            oof_proba = np.zeros((len(X_bundle), label_count), dtype=float)
            fold_scores = []
            fold_samples = []
            for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
                X_tr, X_val = X_bundle[train_idx], X_bundle[val_idx]
                y_tr, y_val = y_bundle[train_idx], y_bundle[val_idx]
                meta_val = {k: v[val_idx] for k, v in meta_bundle.items()}
                source_files_val = source_files_bundle[val_idx]
                w_tr = np.array([weight_dict[label] for label in y_tr])

                model = xgb.XGBClassifier(**best_params)
                model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)
                preds_proba = model.predict_proba(X_val)
                oof_proba[val_idx] = preds_proba
                fold_score, n_assigned = evaluate_view_with_constraints(view, preds_proba, y_val, meta_val, source_files_val)
                fold_scores.append(fold_score)
                fold_samples.append(n_assigned)
                print(f"    Fold {fold}/{n_splits}: assigned={n_assigned}, F1={fold_score:.4f}")

            w_full = np.array([weight_dict[label] for label in y_bundle])
            final_model = xgb.XGBClassifier(**best_params)
            final_model.fit(X_bundle, y_bundle, sample_weight=w_full)
            bundle_models[view] = final_model
            bundle_oof[view] = oof_proba

            mean_score = float(np.mean(fold_scores)) if fold_scores else 0.0
            print(f"    ✓ {dataset_tag} / {view} モデル訓練完了 (mean F1={mean_score:.4f})")

            bundle_view_results.append({
                "dataset_tag": dataset_tag,
                "model": view,
                "mean_f1": mean_score,
                "best_value": float(study.best_value),
                "fold_scores": fold_scores,
                "assigned_samples": fold_samples,
                "best_trial": int(study.best_trial.number),
                "best_params": study.best_params,
            })

        bundle_results.append({
            "dataset_tag": dataset_tag,
            "bundle": bundle,
            "models": bundle_models,
            "oof_proba_by_view": bundle_oof,
            "fold_id_per_row": fold_id_per_row,
            "fold_summary_df": fold_summary_df,
            "fold_role_df": fold_role_df,
            "fold_agent_df": fold_agent_df,
            "view_results": bundle_view_results,
        })
        models_by_bundle[dataset_tag] = bundle_models
        oof_proba_by_bundle[dataset_tag] = bundle_oof
        cv_results.extend(bundle_view_results)

    def build_view_weights(view_name):
        raw_scores = {}
        for result in bundle_results:
            tag = result["dataset_tag"]
            row = next(item for item in result["view_results"] if item["model"] == view_name)
            raw_scores[tag] = max(float(row["mean_f1"]), 1e-6)
        total = sum(raw_scores.values()) or 1.0
        return {tag: score / total for tag, score in raw_scores.items()}

    for view in views:
        ensemble_weights_by_view[view] = build_view_weights(view)

    def ensemble_probs_for_target(target_dataset_tag, view_name):
        target_bundle = next(result for result in bundle_results if result["dataset_tag"] == target_dataset_tag)
        X_target = target_bundle["bundle"]["X"]
        ensemble_proba = np.zeros((len(X_target), len(label_encoder.classes_)), dtype=float)
        for source_result in bundle_results:
            source_tag = source_result["dataset_tag"]
            weight = ensemble_weights_by_view[view_name][source_tag]
            if source_tag == target_dataset_tag:
                proba = source_result["oof_proba_by_view"][view_name]
            else:
                proba = source_result["models"][view_name].predict_proba(X_target)
            ensemble_proba += weight * proba
        return ensemble_proba

    for target_result in bundle_results:
        target_tag = target_result["dataset_tag"]
        bundle = target_result["bundle"]
        for view in views:
            ensemble_proba = ensemble_probs_for_target(target_tag, view)
            score, n_assigned = evaluate_view_with_constraints(view, ensemble_proba, bundle["y"], bundle["meta"], bundle["source_files"])
            ensemble_eval_rows.append({
                "dataset_tag": target_tag,
                "model": view,
                "ensemble_f1": score,
                "assigned_samples": n_assigned,
                "weights": json.dumps(ensemble_weights_by_view[view], ensure_ascii=False),
            })

    ensemble_eval_df = pd.DataFrame(ensemble_eval_rows)
    models = models_by_bundle

    print(f"\n{'=' * 70}")
    print("✓ 全 dataset の個別学習が完了")
    print(f"{'=' * 70}")

    print("\nCV結果サマリー:")
    cv_results_df = pd.DataFrame(cv_results)
    display(cv_results_df[["dataset_tag", "model", "mean_f1", "best_value", "best_trial"]])

    print("\nEnsemble weights by view:")
    for view in views:
        print(f"  {view}: {ensemble_weights_by_view[view]}")

    print("\nEnsemble validation summary:")
    display(ensemble_eval_df)

    run_status["ok"] = True

except Exception as e:
    run_status["error"] = str(e)
    print(f"\nTraining failed: {e}")
    print(traceback.format_exc())
finally:
    run_status["elapsed_sec"] = round(time.time() - t0, 2)

print("\nRun status:")
print(json.dumps(run_status, ensure_ascii=False, indent=2))

1DAY START モデル訓練 (dataset別学習 + OOF ensemble)

✓ dataset bundles: 2

[all_feature_table_2025sp17_with_talks] Fold search completed:
  Best score: 0.262631
  Mean score: 0.332086
  Min score:  0.262631
Fold x Role distribution:


role,POSSESSED,SEER,VILLAGER,WEREWOLF
fold,,,,
1,21,21,42,21
2,22,22,44,22
3,21,21,42,21
4,21,21,42,21
5,21,21,42,21


Fold x Agent distribution (all_feature_table_2025sp17_with_talks):


agent,CamelliaDragons1,CanisLupus1,GAIT1,GPTaku1,ai168wolf1,kanolab1,mille1,osawalab1,sunamelli1
fold,,,,,,,,,
1,10,12,10,11,14,12,11,15,10
2,12,13,9,10,12,15,12,15,12
3,11,12,10,13,12,15,10,11,11
4,9,11,8,13,15,14,12,13,10
5,10,12,9,10,10,16,13,11,14



[all_feature_table_2025sp17_with_talks] training start

  all_feature_table_2025sp17_with_talks / SEER 視点モデル: Optuna探索開始

    Best trial: 11
    Best CV score: 0.6407
    Best params: {'max_depth': 9, 'learning_rate': 0.04896940100699438, 'n_estimators': 377, 'min_child_weight': 12, 'subsample': 0.7518138484340006, 'colsample_bytree': 0.9789880874538218, 'gamma': 0.03669849229776129, 'reg_alpha': 8.28720193487304, 'reg_lambda': 9.710662705281718}
    Fold 1/5: assigned=84, F1=0.6190
    Fold 2/5: assigned=88, F1=0.7273
    Fold 3/5: assigned=84, F1=0.6190
    Fold 4/5: assigned=84, F1=0.5714
    Fold 5/5: assigned=84, F1=0.6667
    ✓ all_feature_table_2025sp17_with_talks / SEER モデル訓練完了 (mean F1=0.6407)

  all_feature_table_2025sp17_with_talks / WEREWOLF 視点モデル: Optuna探索開始

    Best trial: 17
    Best CV score: 0.4351
    Best params: {'max_depth': 4, 'learning_rate': 0.0369417370601914, 'n_estimators': 538, 'min_child_weight': 3, 'subsample': 0.6648115337812508, 'colsample_bytree': 0.7

...


[all_feature_table_2025summer_with_talks2] training start

  all_feature_table_2025summer_with_talks2 / SEER 視点モデル: Optuna探索開始

    Best trial: 11
    Best CV score: 0.5900
    Best params: {'max_depth': 3, 'learning_rate': 0.01103616456973904, 'n_estimators': 381, 'min_child_weight': 8, 'subsample': 0.7171781358782057, 'colsample_bytree': 0.865042426595599, 'gamma': 0.0033749586084441826, 'reg_alpha': 0.0029151336209232927, 'reg_lambda': 1.9082322636172706e-08}
    Fold 1/5: assigned=96, F1=0.6250
    Fold 2/5: assigned=88, F1=0.4091
    Fold 3/5: assigned=84, F1=0.7143
    Fold 4/5: assigned=88, F1=0.6364
    Fold 5/5: assigned=92, F1=0.5652
    ✓ all_feature_table_2025summer_with_talks2 / SEER モデル訓練完了 (mean F1=0.5900)

  all_feature_table_2025summer_with_talks2 / WEREWOLF 視点モデル: Optuna探索開始

    Best trial: 15
    Best CV score: 0.4721
    Best params: {'max_depth': 5, 'learning_rate': 0.03519567525394583, 'n_estimators': 289, 'min_child_weight': 8, 'subsample': 0.7942900825189663, 

,dataset_tag,model,mean_f1,best_value,best_trial
0,all_feature_table_2025sp17_with_talks,SEER,0.640693,0.640693,11
1,all_feature_table_2025sp17_with_talks,WEREWOLF,0.435065,0.435065,17
2,all_feature_table_2025sp17_with_talks,VILLAGER,0.414286,0.414286,19
3,all_feature_table_2025sp17_with_talks,POSSESSED,0.367100,0.367100,3
4,all_feature_table_2025summer_with_talks2,SEER,0.589992,0.589992,11
5,all_feature_table_2025summer_with_talks2,WEREWOLF,0.472101,0.472101,15
6,all_feature_table_2025summer_with_talks2,VILLAGER,0.449094,0.449094,14
7,all_feature_table_2025summer_with_talks2,POSSESSED,0.412319,0.412319,9



Ensemble weights by view:
  SEER: {'all_feature_table_2025sp17_with_talks': 0.5205987497371389, 'all_feature_table_2025summer_with_talks2': 0.47940125026286107}
  WEREWOLF: {'all_feature_table_2025sp17_with_talks': 0.47958670269877746, 'all_feature_table_2025summer_with_talks2': 0.5204132973012225}
  VILLAGER: {'all_feature_table_2025sp17_with_talks': 0.47984173136296876, 'all_feature_table_2025summer_with_talks2': 0.5201582686370312}
  POSSESSED: {'all_feature_table_2025sp17_with_talks': 0.470991656705425, 'all_feature_table_2025summer_with_talks2': 0.5290083432945751}

Ensemble validation summary:


,dataset_tag,model,ensemble_f1,assigned_samples,weights
0,all_feature_table_2025sp17_with_talks,SEER,0.613208,424,"{""all_feature_table_2025sp17_with_talks"": 0.52..."
1,all_feature_table_2025sp17_with_talks,WEREWOLF,0.405660,424,"{""all_feature_table_2025sp17_with_talks"": 0.47..."
2,all_feature_table_2025sp17_with_talks,VILLAGER,0.448113,848,"{""all_feature_table_2025sp17_with_talks"": 0.47..."
3,all_feature_table_2025sp17_with_talks,POSSESSED,0.367925,424,"{""all_feature_table_2025sp17_with_talks"": 0.47..."
4,all_feature_table_2025summer_with_talks2,SEER,0.580357,448,"{""all_feature_table_2025sp17_with_talks"": 0.52..."
5,all_feature_table_2025summer_with_talks2,WEREWOLF,0.378151,476,"{""all_feature_table_2025sp17_with_talks"": 0.47..."
6,all_feature_table_2025summer_with_talks2,VILLAGER,0.390756,952,"{""all_feature_table_2025sp17_with_talks"": 0.47..."
7,all_feature_table_2025summer_with_talks2,POSSESSED,0.386555,476,"{""all_feature_table_2025sp17_with_talks"": 0.47..."



Run status:
{
  "ok": true,
  "error": null,
  "elapsed_sec": 789.32
}


In [10]:
print("=" * 80)
print("固定重み 0.5 の比較評価")
print("=" * 80)

if 'bundle_results' not in globals() or 'models_by_bundle' not in globals():
    raise RuntimeError('先に学習セルを実行してください。')

if len(bundle_results) != 2:
    print(f"warning: bundle_results={len(bundle_results)} ですが、ここでは固定 0.5 を 2 データセット前提で評価します。")

fixed_dataset_tags = [result["dataset_tag"] for result in bundle_results]
fixed_weights_by_view = {view: {tag: 0.5 for tag in fixed_dataset_tags} for view in views}


def ensemble_probs_for_target_fixed(target_dataset_tag, view_name):
    target_bundle = next(result for result in bundle_results if result["dataset_tag"] == target_dataset_tag)
    X_target = target_bundle["bundle"]["X"]
    ensemble_proba = np.zeros((len(X_target), len(label_encoder.classes_)), dtype=float)

    for source_result in bundle_results:
        source_tag = source_result["dataset_tag"]
        if source_tag == target_dataset_tag:
            proba = source_result["oof_proba_by_view"][view_name]
        else:
            proba = source_result["models"][view_name].predict_proba(X_target)
        ensemble_proba += 0.5 * proba

    return ensemble_proba


fixed_eval_rows = []
for target_result in bundle_results:
    target_tag = target_result["dataset_tag"]
    bundle = target_result["bundle"]
    for view in views:
        ensemble_proba = ensemble_probs_for_target_fixed(target_tag, view)
        score, n_assigned = evaluate_view_with_constraints(view, ensemble_proba, bundle["y"], bundle["meta"], bundle["source_files"])
        fixed_eval_rows.append({
            "dataset_tag": target_tag,
            "model": view,
            "ensemble_f1_fixed05": score,
            "assigned_samples": n_assigned,
            "weights": json.dumps(fixed_weights_by_view[view], ensure_ascii=False),
        })

fixed_ensemble_eval_df = pd.DataFrame(fixed_eval_rows)
display(fixed_ensemble_eval_df)

if 'ensemble_eval_df' in globals() and isinstance(ensemble_eval_df, pd.DataFrame) and not ensemble_eval_df.empty:
    compare_df = fixed_ensemble_eval_df.merge(
        ensemble_eval_df[["dataset_tag", "model", "ensemble_f1"]],
        on=["dataset_tag", "model"],
        how="left",
    )
    compare_df["delta_fixed05_minus_current"] = compare_df["ensemble_f1_fixed05"] - compare_df["ensemble_f1"]
    display(compare_df)

print("固定 0.5 のみの比較評価が完了しました。")

固定重み 0.5 の比較評価


,dataset_tag,model,ensemble_f1_fixed05,assigned_samples,weights
0,all_feature_table_2025sp17_with_talks,SEER,0.613208,424,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
1,all_feature_table_2025sp17_with_talks,WEREWOLF,0.405660,424,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
2,all_feature_table_2025sp17_with_talks,VILLAGER,0.438679,848,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
3,all_feature_table_2025sp17_with_talks,POSSESSED,0.358491,424,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
4,all_feature_table_2025summer_with_talks2,SEER,0.589286,448,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
5,all_feature_table_2025summer_with_talks2,WEREWOLF,0.378151,476,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
6,all_feature_table_2025summer_with_talks2,VILLAGER,0.390756,952,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."
7,all_feature_table_2025summer_with_talks2,POSSESSED,0.394958,476,"{""all_feature_table_2025sp17_with_talks"": 0.5,..."


,dataset_tag,model,ensemble_f1_fixed05,assigned_samples,weights,ensemble_f1,delta_fixed05_minus_current
0,all_feature_table_2025sp17_with_talks,SEER,0.613208,424,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.613208,0.000000
1,all_feature_table_2025sp17_with_talks,WEREWOLF,0.405660,424,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.405660,0.000000
2,all_feature_table_2025sp17_with_talks,VILLAGER,0.438679,848,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.448113,-0.009434
3,all_feature_table_2025sp17_with_talks,POSSESSED,0.358491,424,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.367925,-0.009434
4,all_feature_table_2025summer_with_talks2,SEER,0.589286,448,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.580357,0.008929
5,all_feature_table_2025summer_with_talks2,WEREWOLF,0.378151,476,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.378151,0.000000
6,all_feature_table_2025summer_with_talks2,VILLAGER,0.390756,952,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.390756,0.000000
7,all_feature_table_2025summer_with_talks2,POSSESSED,0.394958,476,"{""all_feature_table_2025sp17_with_talks"": 0.5,...",0.386555,0.008403


固定 0.5 のみの比較評価が完了しました。


In [11]:
print("=" * 80)
print("データセット全体の加重平均スコア")
print("=" * 80)

if 'fixed_ensemble_eval_df' not in globals():
    raise RuntimeError('先に固定 0.5 の比較評価セルを実行してください。')

if 'ensemble_eval_df' not in globals():
    raise RuntimeError('先に学習・評価セルを実行してください。')

# データセットごとの ensemble_f1 を assigned_samples で加重平均
weighted_current_by_view = (
    ensemble_eval_df
    .groupby('model', as_index=False)
    .apply(lambda frame: pd.Series({
        'weighted_ensemble_f1': np.average(frame['ensemble_f1'], weights=frame['assigned_samples'])
    }))
    .reset_index(drop=True)
)

weighted_fixed_by_view = (
    fixed_ensemble_eval_df
    .groupby('model', as_index=False)
    .apply(lambda frame: pd.Series({
        'weighted_ensemble_f1_fixed05': np.average(frame['ensemble_f1_fixed05'], weights=frame['assigned_samples'])
    }))
    .reset_index(drop=True)
)

weighted_compare_df = weighted_current_by_view.merge(weighted_fixed_by_view, on='model', how='outer')
weighted_compare_df['delta_fixed05_minus_current'] = (
    weighted_compare_df['weighted_ensemble_f1_fixed05'] - weighted_compare_df['weighted_ensemble_f1']
)

display(weighted_compare_df)

# 全モデルをまとめた加重平均
current_overall = np.average(ensemble_eval_df['ensemble_f1'], weights=ensemble_eval_df['assigned_samples'])
fixed_overall = np.average(fixed_ensemble_eval_df['ensemble_f1_fixed05'], weights=fixed_ensemble_eval_df['assigned_samples'])

summary_df = pd.DataFrame([
    {
        'metric': 'overall_weighted_ensemble_f1',
        'value': current_overall,
    },
    {
        'metric': 'overall_weighted_ensemble_f1_fixed05',
        'value': fixed_overall,
    },
    {
        'metric': 'delta_fixed05_minus_current',
        'value': fixed_overall - current_overall,
    },
])
display(summary_df)

print('加重平均スコアを算出しました。')

データセット全体の加重平均スコア


,model,weighted_ensemble_f1,weighted_ensemble_f1_fixed05,delta_fixed05_minus_current
0,POSSESSED,0.377778,0.377778,0.000000
1,SEER,0.596330,0.600917,0.004587
2,VILLAGER,0.417778,0.413333,-0.004444
3,WEREWOLF,0.391111,0.391111,0.000000


,metric,value
0,overall_weighted_ensemble_f1,0.439177
1,overall_weighted_ensemble_f1_fixed05,0.438283
2,delta_fixed05_minus_current,-0.000894


加重平均スコアを算出しました。


In [5]:
# モデル別の最適パラメータを見やすく表示
if 'cv_results_df' in globals() and isinstance(cv_results_df, pd.DataFrame) and not cv_results_df.empty:
    rows = []
    for r in cv_results_df.to_dict(orient='records'):
        row = {
            'dataset_tag': r.get('dataset_tag'),
            'model': r.get('model'),
            'best_value': r.get('best_value'),
            'best_trial': r.get('best_trial'),
        }
        best_params = r.get('best_params', {})
        if isinstance(best_params, dict):
            row.update(best_params)
        rows.append(row)
    best_params_df = pd.DataFrame(rows)
    display(best_params_df)
else:
    print('no results')

no results


In [11]:
EXPERIMENT_TAG = globals().get("EXPERIMENT_TAG", "baseline")
SPLIT_VERSION = globals().get("SPLIT_VERSION", "dataset_plus_source")

print("=" * 80)
print("Experiment tracking")
print("=" * 80)
print(f"EXPERIMENT_TAG : {EXPERIMENT_TAG}")
print(f"SPLIT_VERSION  : {SPLIT_VERSION}")
print("このタグ名で保存ファイルが分離されます。")

Experiment tracking
EXPERIMENT_TAG : baseline
SPLIT_VERSION  : dataset_plus_source
このタグ名で保存ファイルが分離されます。


In [ ]:
# 特徴量重要度（上位k）
feature_rows = []
if 'models_by_bundle' in globals() and models_by_bundle:
    for dataset_tag, view_models in models_by_bundle.items():
        for model_name, model in view_models.items():
            imp = model.get_booster().get_score(importance_type='weight')
            top_items = sorted(imp.items(), key=lambda x: x[1], reverse=True)[: RUN_OPTIONS['top_k_features']]
            total = sum(v for _, v in top_items) or 1.0
            for rank, (feat, val) in enumerate(top_items, start=1):
                feature_rows.append(
                    {
                        'dataset_tag': dataset_tag,
                        'model': model_name,
                        'rank': rank,
                        'feature': feat,
                        'importance': val,
                        'importance_ratio': val / total,
                    }
                )

fi_df = pd.DataFrame(feature_rows)
fi_df.head(20)

,model,rank,feature,importance,importance_ratio
0,SEER,1,f67,399.0,0.278243
1,SEER,2,f15,168.0,0.117155
2,SEER,3,f71,155.0,0.108089
3,SEER,4,f0,125.0,0.087169
4,SEER,5,f31,111.0,0.077406
5,SEER,6,f187,104.0,0.072524
6,SEER,7,f39,100.0,0.069735
7,SEER,8,f79,94.0,0.065551
8,SEER,9,f81,93.0,0.064854
9,SEER,10,f27,85.0,0.059275


In [ ]:
import shutil

print("=" * 80)
print("結果の保存")
print("=" * 80)

base_output_dir = PROJECT_ROOT / "notebooks" / "outputs" / "1day_start_individual_ensemble"
base_output_dir.mkdir(parents=True, exist_ok=True)

exp_tag = globals().get("EXPERIMENT_TAG", "baseline")
split_version = globals().get("SPLIT_VERSION", "dataset_individual_plus_ensemble")
safe_tag = str(exp_tag).replace(" ", "_")
exp_dir = base_output_dir / "experiments" / safe_tag
exp_dir.mkdir(parents=True, exist_ok=True)

def tagged_name(stem, ext):
    return f"{stem}_{safe_tag}.{ext}"

saved_model_name = tagged_name("models_1day_start_individual_ensemble", "joblib")
saved_model_path = exp_dir / saved_model_name

if 'models_by_bundle' in globals() and models_by_bundle:
    base_model_path = base_output_dir / "models_1day_start_individual_ensemble.joblib"
    joblib.dump(models_by_bundle, base_model_path)
    joblib.dump(models_by_bundle, saved_model_path)
    print(f"✓ Models saved: {base_model_path}")
    print(f"✓ Tagged models saved: {saved_model_path}")

    for dataset_tag, model_dict in models_by_bundle.items():
        dataset_model_path = exp_dir / tagged_name(f"models_1day_start_{dataset_tag}", "joblib")
        joblib.dump(model_dict, dataset_model_path)
        print(f"✓ Dataset models saved: {dataset_model_path}")
else:
    print("保存対象のモデルがありません。先に学習セルを実行してください。")

if 'fi_df' in globals() and isinstance(fi_df, pd.DataFrame) and not fi_df.empty:
    base_fi_path = base_output_dir / "feature_importance_1day_start_individual_ensemble.csv"
    fi_df.to_csv(base_fi_path, index=False)
    fi_tagged_path = exp_dir / tagged_name("feature_importance", "csv")
    fi_df.to_csv(fi_tagged_path, index=False)
    print(f"✓ Feature importance saved: {base_fi_path}")
    print(f"✓ Tagged feature importance saved: {fi_tagged_path}")

if 'bundle_results' in globals() and bundle_results:
    bundle_results_path = base_output_dir / "bundle_results_1day_start_individual_ensemble.joblib"
    joblib.dump(bundle_results, bundle_results_path)
    print(f"✓ Bundle results saved: {bundle_results_path}")

    for result in bundle_results:
        dataset_tag = result["dataset_tag"]
        fold_summary_df = result.get("fold_summary_df")
        fold_role_df = result.get("fold_role_df")
        fold_agent_df = result.get("fold_agent_df")

        if isinstance(fold_summary_df, pd.DataFrame):
            path = exp_dir / tagged_name(f"fold_split_basic_info_{dataset_tag}", "csv")
            fold_summary_df.to_csv(path, index=False)
            print(f"✓ Fold split summary saved: {path}")

        if isinstance(fold_role_df, pd.DataFrame):
            path = exp_dir / tagged_name(f"fold_role_distribution_{dataset_tag}", "csv")
            fold_role_df.to_csv(path, index=False)
            print(f"✓ Fold role distribution saved: {path}")

        if isinstance(fold_agent_df, pd.DataFrame):
            path = exp_dir / tagged_name(f"fold_agent_distribution_{dataset_tag}", "csv")
            fold_agent_df.to_csv(path, index=False)
            print(f"✓ Fold agent distribution saved: {path}")

if 'cv_results_df' in globals() and isinstance(cv_results_df, pd.DataFrame) and not cv_results_df.empty:
    base_cv_json_path = base_output_dir / "cv_results_1day_start_individual_ensemble.json"
    cv_results_df.to_json(base_cv_json_path, orient="records", force_ascii=False, indent=2)
    print(f"✓ CV results saved: {base_cv_json_path}")

    cv_summary_df = pd.DataFrame([
        {
            "dataset_tag": r.get("dataset_tag"),
            "model": r.get("model"),
            "mean_f1": r.get("mean_f1"),
            "best_value": r.get("best_value"),
            "best_trial": r.get("best_trial"),
            "split_version": split_version,
            "experiment_tag": exp_tag,
        }
        for r in cv_results
        if isinstance(r, dict)
    ])
    base_cv_summary_path = base_output_dir / "cv_summary_1day_start_individual_ensemble.csv"
    cv_summary_df.to_csv(base_cv_summary_path, index=False)
    print(f"✓ CV summary saved: {base_cv_summary_path}")

    tag_cv_path = exp_dir / tagged_name("cv_summary", "csv")
    cv_summary_df.to_csv(tag_cv_path, index=False)
    print(f"✓ Tagged cv_summary saved: {tag_cv_path}")
else:
    print("cv_results_df がないため CV 出力は保存しません")

if 'ensemble_eval_df' in globals() and isinstance(ensemble_eval_df, pd.DataFrame) and not ensemble_eval_df.empty:
    ensemble_eval_path = base_output_dir / "ensemble_eval_1day_start_individual_ensemble.csv"
    ensemble_eval_df.to_csv(ensemble_eval_path, index=False)
    print(f"✓ Ensemble eval saved: {ensemble_eval_path}")

if 'ensemble_weights_by_view' in globals():
    weights_path = base_output_dir / "ensemble_weights_1day_start_individual_ensemble.json"
    with open(weights_path, "w", encoding="utf-8") as f:
        json.dump(ensemble_weights_by_view, f, ensure_ascii=False, indent=2)
    print(f"✓ Ensemble weights saved: {weights_path}")

if 'bundle_summary_df' in globals() and isinstance(bundle_summary_df, pd.DataFrame):
    base_bundle_summary_path = base_output_dir / "bundle_summary_1day_start_individual_ensemble.csv"
    bundle_summary_df.to_csv(base_bundle_summary_path, index=False)
    print(f"✓ Bundle summary saved: {base_bundle_summary_path}")

meta_out = {
    "generated_at": datetime.now().isoformat(),
    "day_filter": RUN_OPTIONS["day_filter"],
    "day2_flag": RUN_OPTIONS["day2_flag"],
    "n_trials": RUN_OPTIONS["n_trials"],
    "n_samples": int(sum(len(bundle["X"]) for bundle in dataset_bundles)) if 'dataset_bundles' in globals() else int(len(X)),
    "n_features": int(len(final_features)) if 'final_features' in globals() else None,
    "status": run_status,
    "experiment_tag": exp_tag,
    "split_version": split_version,
    "saved_model_file": saved_model_name,
    "dataset_tags": [bundle["dataset_tag"] for bundle in dataset_bundles] if 'dataset_bundles' in globals() else [],
    "ensemble_weights_by_view": ensemble_weights_by_view if 'ensemble_weights_by_view' in globals() else {},
}
base_meta_path = base_output_dir / "metadata_1day_start_individual_ensemble.json"
with open(base_meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_out, f, ensure_ascii=False, indent=2)
print(f"✓ Metadata saved: {base_meta_path}")

meta_path = exp_dir / tagged_name("experiment_meta", "json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_out, f, ensure_ascii=False, indent=2)
print(f"✓ Tagged experiment_meta saved: {meta_path}")

print(f"\nSaved directory: {base_output_dir}")
print(f"Tagged directory: {exp_dir}")

結果の保存
✓ Models saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\models_1day_start.joblib
✓ Tagged models saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\experiments\baseline\models_1day_start_baseline.joblib
✓ CV results saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\cv_results_1day_start.json
✓ CV summary saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\cv_summary_1day_start.csv
✓ Tagged cv_summary saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\experiments\baseline\cv_summary_baseline.csv
✓ Metadata saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\metadata_1day_start.json
✓ Tagged experiment_meta saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\experiments\baseline\experiment_meta_baseline.json

Sa